## KNN Recommendation Model

In [1]:
# import
import numpy as np
import pandas as pd
import requests
from io import BytesIO
from PIL import Image

import torch
import torch.nn as nn

from sklearn.neighbors import NearestNeighbors
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

In [2]:
df = pd.read_csv('data/final_df.csv')
df.head()

,Unnamed: 0,filename,link,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image_data
0,0,1638,http://assets.myntassets.com/v1/images/style/p...,1638,Unisex,Apparel,Bottomwear,Swimwear,Blue,Fall,2010.0,Sports,Nabaiji Swimming Goggles Blue Black,<PIL.Image.Image image mode=RGB size=1800x2400...
1,1,32903,http://assets.myntassets.com/v1/images/style/p...,32903,Women,Apparel,Bottomwear,Shorts,Red,Summer,2012.0,Casual,Allen Solly Women Red Shorts,<PIL.Image.Image image mode=RGB size=1080x1440...
2,2,39381,http://assets.myntassets.com/v1/images/style/p...,39381,Men,Apparel,Bottomwear,Jeans,Black,Summer,2012.0,Casual,Peter England Men Party Black Jeans,<PIL.Image.Image image mode=RGB size=1800x2400...
3,3,12163,http://assets.myntassets.com/v1/images/style/p...,12163,Women,Apparel,Bottomwear,Leggings,Purple,Fall,2011.0,Ethnic,Aurelia Women Solid Purple Leggings,<PIL.Image.Image image mode=RGB size=1800x2400...
4,4,1607,http://assets.myntassets.com/v1/images/style/p...,1607,Men,Apparel,Bottomwear,Track Pants,Blue,Fall,2010.0,Sports,Reebok Men trackpant- male Track Pants,<PIL.Image.Image image mode=RGB size=1080x1440...


In [3]:
# dataframe
print(df.columns)
print(df.shape)

Index(['Unnamed: 0', 'filename', 'link', 'id', 'gender', 'masterCategory',
       'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage',
       'productDisplayName', 'image_data'],
      dtype='object')
(1500, 14)


In [4]:
# image preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])])

In [ ]:
# combined label
df["img_label"] = df["subCategory"] + " " + df["season"] + " " + df["gender"]
df[["subCategory", "season", "gender", "img_label"]].head()

,subCategory,season,gender,img_label
0,Bottomwear,Fall,Unisex,Bottomwear Fall Unisex
1,Bottomwear,Summer,Women,Bottomwear Summer Women
2,Bottomwear,Summer,Men,Bottomwear Summer Men
3,Bottomwear,Fall,Women,Bottomwear Fall Women
4,Bottomwear,Fall,Men,Bottomwear Fall Men


In [8]:
# iamge transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])])

In [ ]:
# set device
# using pretrained ResNet - will swap later
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

weights = ResNet18_Weights.DEFAULT
base_model = resnet18(weights=weights)

feature_extractor = nn.Sequential(*list(base_model.children())[:-1])
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

cpu


Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Con

In [12]:
# load one image from URL
def load_image_from_url(url):
    response = requests.get(url, timeout=10)
    img = Image.open(BytesIO(response.content)).convert("RGB")
    return img

In [13]:
# extract features from one image
def get_image_feature(url):
    try:
        img = load_image_from_url(url)
        img_tensor = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            feature = feature_extractor(img_tensor)

        return feature.cpu().numpy().flatten()

    except Exception as e:
        print(f"Error loading image: {url}")
        return None

In [15]:
# test one image
sample_feature = get_image_feature(df.loc[0, "link"])
print(type(sample_feature))
print(sample_feature.shape)

<class 'numpy.ndarray'>
(512,)


In [21]:
print(df["link"].head(20))
print(df["link"].dtype)
print(df["link"].isna().sum())
print((df["link"] == "undefined").sum())

0     http://assets.myntassets.com/v1/images/style/p...
1     http://assets.myntassets.com/v1/images/style/p...
2     http://assets.myntassets.com/v1/images/style/p...
3     http://assets.myntassets.com/v1/images/style/p...
4     http://assets.myntassets.com/v1/images/style/p...
5     http://assets.myntassets.com/v1/images/style/p...
6     http://assets.myntassets.com/v1/images/style/p...
7     http://assets.myntassets.com/v1/images/style/p...
8     http://assets.myntassets.com/v1/images/style/p...
9     http://assets.myntassets.com/v1/images/style/p...
10    http://assets.myntassets.com/v1/images/style/p...
11    http://assets.myntassets.com/v1/images/style/p...
12    http://assets.myntassets.com/v1/images/style/p...
13    http://assets.myntassets.com/v1/images/style/p...
14    http://assets.myntassets.com/v1/images/style/p...
15    http://assets.myntassets.com/v1/images/style/p...
16    http://assets.myntassets.com/v1/images/style/p...
17    http://assets.myntassets.com/v1/images/sty

In [22]:
df = df[df["link"].notna()].copy()
df = df[df["link"] != "undefined"].copy()
df = df[df["link"].str.startswith("http", na=False)].copy()

df = df.reset_index(drop=True)

print(df.shape)
print(df["link"].head())

(1499, 15)
0    http://assets.myntassets.com/v1/images/style/p...
1    http://assets.myntassets.com/v1/images/style/p...
2    http://assets.myntassets.com/v1/images/style/p...
3    http://assets.myntassets.com/v1/images/style/p...
4    http://assets.myntassets.com/v1/images/style/p...
Name: link, dtype: object


In [23]:
def get_image_feature(url):
    try:
        if pd.isna(url) or url == "undefined" or not str(url).startswith("http"):
            return None

        img = load_image_from_url(url)
        img_tensor = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            feature = feature_extractor(img_tensor)

        return feature.cpu().numpy().flatten()

    except Exception as e:
        print(f"Error loading image: {url}")
        return None

In [24]:
feature_list = []
valid_rows = []

for i in range(len(df)):
    if i % 100 == 0:
        print(f"Processing image {i} of {len(df)}")

    url = df.loc[i, "link"]
    feature = get_image_feature(url)

    if feature is not None:
        feature_list.append(feature)
        valid_rows.append(i)

Processing image 0 of 1499
Processing image 100 of 1499
Processing image 200 of 1499
Processing image 300 of 1499
Processing image 400 of 1499
Processing image 500 of 1499
Processing image 600 of 1499
Processing image 700 of 1499
Processing image 800 of 1499
Processing image 900 of 1499
Processing image 1000 of 1499
Processing image 1100 of 1499
Processing image 1200 of 1499
Processing image 1300 of 1499
Processing image 1400 of 1499


In [25]:
# build feature matrix
features_array = np.array(feature_list)
df_knn = df.iloc[valid_rows].reset_index(drop=True)

print("Feature matrix shape:", features_array.shape)
print("Filtered dataframe shape:", df_knn.shape)

Feature matrix shape: (1499, 512)
Filtered dataframe shape: (1499, 15)
